# Init

In [0]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd

# Lấy dữ liệu và tạo cột 

In [0]:
df_ml_raw = spark.table("workspace.silver.lc_loans").filter(
    col("loan_status").isin("Fully Paid", "Charged Off")
)

df_ml = df_ml_raw.withColumn(
    "label", 
    when(col("loan_status") == "Charged Off", 1).otherwise(0)
)

# Chọn lọc các biến đầu vào

In [0]:
num_cols = [
    "loan_amount",               
    "interest_rate",              
    "installment", 
    "annual_income",              
    "dti", 
    "employment_length_years",    
    "delinquencies_past_2yrs",    
    "open_credit_lines",
    "revolving_utilization_rate",  
    "credit_inquiries_last_6mths", 
    "total_credit_lines",          
    "public_records_derogatory"           
]

cat_cols = [
    "home_ownership", 
    "loan_purpose",               
    "verification_status",
    "grade"
]

df_ml = df_ml.select(["label"] + num_cols + cat_cols)

# Chuẩn bị các biến cần thiết cho AI (Feature Engineering)

In [0]:
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in cat_cols]

assembler_inputs = [f"{c}_ohe" for c in cat_cols] + num_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="skip")

# Khai báo Thuật toán và cho AI "HỌC"

In [0]:


rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=250, maxDepth=11, minInstancesPerNode=5)

ml_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

df_ngoan = df_ml.filter(col("label") == 0)
df_hu = df_ml.filter(col("label") == 1)

so_hu = df_hu.count()

df_ngoan_downsampled = df_ngoan.sample(withReplacement=False, fraction=(so_hu * 1.5) / df_ngoan.count(), seed=42)

df_ml_balanced = df_ngoan_downsampled.unionAll(df_hu)

train_df, test_df = df_ml_balanced.randomSplit([0.8, 0.2], seed=42)

trained_model = ml_pipeline.fit(train_df)

# Test AI (Evaluation)

In [0]:
predictions = trained_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_score = evaluator.evaluate(predictions)

print(f"Điểm ROC-AUC của AI: {auc_score:.4f}")

In [0]:
sample_df = trained_model.transform(test_df.limit(1))

meta = sample_df.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in meta:
        all_features.extend(meta[attr_type])

all_features.sort(key=lambda x: x["idx"])
exact_feature_names = [item["name"] for item in all_features]

importances = trained_model.stages[-1].featureImportances.toArray()

df_importance = pd.DataFrame(
    {"Feature": exact_feature_names, "Importance": importances}
).sort_values(by="Importance", ascending=False)

print(
    f"Khớp lệnh! Bộ não AI đang vận hành tổng cộng {len(exact_feature_names)} nơ-ron thông tin.\n"
)
print("TOP 10 YẾU TỐ KHIẾN KHÁCH HÀNG BÙNG NỢ:")
print(df_importance.head(10).to_string(index=False))